# PND Thesis — GPU battery on Kaggle (resumable, multi-session)

Runs 37 jobs (fine-tune at_pre / at_scratch / cnnbilstm × 3 seeds, label-efficiency at 10/25/50% of events, B7/B10 unsupervised) **in priority order**, Gate 3 evidence first.

**How sessions chain:**
1. Settings: Accelerator **GPU T4 x2** (two jobs in parallel), Internet **ON**. Inputs: `pnd-thesis-processed` + `pnd-thesis-checkpoint`.
2. *Save Version → Save & Run All (Commit)*. The queue stops itself before `BUDGET_HOURS`, saving partial state; nothing finished is ever redone.
3. If jobs are left: **Add Input → Your Work → this notebook's output (latest version)**, then commit again. Step 5 restores everything automatically.
4. Repeat until Step 5 prints `0 pending`. Download `pnd-results.zip` from the final version's Output.

In [ ]:
# ── Step 1: Clone latest code ───────────────────────────────────────────────
!rm -rf /kaggle/working/thesis
!git clone -q https://github.com/thinhpd1906/deteching_pump_dump_crypto_real_time.git /kaggle/working/thesis
%cd /kaggle/working/thesis
!git log --oneline -3

In [ ]:
# ── Step 2: Environment (Kaggle image already has torch/sklearn/pandas/yaml;
#    skipping requirements.txt avoids installing pyspark/kafka we don't need) ─
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| GPUs:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(' ', i, torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), 'No GPU -- set Accelerator to GPU T4 x2'
BUDGET_HOURS = 8.5   # stop launching/continuing work after this; raise to ~11 if your session limit is 12h

In [ ]:
# ── Step 3: Locate processed data under /kaggle/input (any slug / layout) ──
import os, shutil, zipfile, glob
import numpy as np

os.makedirs('data/processed', exist_ok=True)
hits = [h for h in glob.glob('/kaggle/input/**/adaptive/train.npz', recursive=True)]
if hits:
    root = os.path.dirname(os.path.dirname(hits[0]))
    print('Found unzipped data at', root)
    shutil.copytree(root, 'data/processed', dirs_exist_ok=True)
else:
    zips = glob.glob('/kaggle/input/**/pnd-processed.zip', recursive=True)
    assert zips, 'processed data not found under /kaggle/input -- attach the dataset'
    print('Extracting', zips[0])
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall('data')          # zip entries start with processed/<mode>/...

for mode in ['none', 'static', 'adaptive']:
    path = f'data/processed/{mode}/train.npz'
    assert os.path.exists(path), f'{path} missing -- check layout above'
    d = np.load(path)
    print(f'{mode}: train={len(d["y"]):,} windows, pos={int(d["y"].sum()):,}')

In [ ]:
# ── Step 4: Pretrained checkpoint ──────────────────────────────────────────
os.makedirs('artifacts/adaptive', exist_ok=True)
ck = [c for c in glob.glob('/kaggle/input/**/at_pretrained.pt', recursive=True)]
assert ck, 'at_pretrained.pt not found -- attach pnd-thesis-checkpoint'
shutil.copy(ck[0], 'artifacts/adaptive/at_pretrained.pt')
print('Checkpoint loaded from', ck[0])

In [ ]:
# ── Step 5: Run the queue (restores previous sessions' results first) ──────
!python -u scripts/kaggle_queue.py --clean adaptive --restore /kaggle/input --budget-hours {BUDGET_HOURS}

In [ ]:
# ── Step 6: Aggregate whatever is done so far ──────────────────────────────
!python -u -m src.training.label_efficiency --clean adaptive --seeds 3
!python -u -m src.training.evaluate --clean adaptive

In [ ]:
# ── Step 7: Keep output small + package results ────────────────────────────
shutil.rmtree('data/processed', ignore_errors=True)   # 450MB, re-attached each session
out = '/kaggle/working/pnd-results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob('artifacts/adaptive/*.json') + glob.glob('artifacts/adaptive/logs/*.log'):
        zf.write(f)
print(f'{out}: {os.path.getsize(out)/1e6:.1f} MB')
!python scripts/kaggle_queue.py --clean adaptive --list | head -3